# JupySQL ADBC Arrow results

This notebook compares JupySQL's existing SQLAlchemy result path with an opt-in [ADBC](https://arrow.apache.org/adbc/) path that reads results as Arrow record batches. `.PolarsDataFrame()` and `.ArrowTable()` then skip row-by-row conversion. The engine used here is DuckDB: the `duckdb` package ships the ADBC driver (`adbc_driver_duckdb`).

A reproducible prototype, patches, and the full local benchmark write-up are in [aoelvp94/jupysql-adbc-demo](https://github.com/aoelvp94/jupysql-adbc-demo). This is a personal demo, not an official JupySQL change.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aoelvp94/jupysql-adbc-demo/blob/main/colab/jupysql_adbc_colab.ipynb)

Colab hardware is shared (typically 2 vCPU), so the timings here will differ from the numbers in the [demo README](https://github.com/aoelvp94/jupysql-adbc-demo), which were measured on an 8 vCPU, 15 GB RAM Linux box.


In [ ]:
import os

JUPYSQL_INSTALL = os.environ.get(
    "JUPYSQL_INSTALL",
    "git+https://github.com/aoelvp94/jupysql@adbc-arrow-results",
)
print("Installing jupysql from:", JUPYSQL_INSTALL)

%pip install -q -U {JUPYSQL_INSTALL} adbc-driver-manager polars pyarrow duckdb duckdb-engine sqlalchemy matplotlib pytz


## Datasets

The tables match `setup_data.py` in the demo repository: narrow `(BIGINT, DOUBLE)` and wide mixed-type rows, at 100k and 1M rows.


In [ ]:
import duckdb

NARROW = (
    "SELECT g::BIGINT AS id, (g * 0.37)::DOUBLE AS x "
    "FROM generate_series(1, {n}) AS t(g)"
)
WIDE = """
SELECT g::BIGINT AS id,
       (g % 1000)::INTEGER AS i,
       (g * 0.37)::DOUBLE AS f,
       ((g % 100000) / 7.0)::DECIMAL(18, 4) AS dec,
       (TIMESTAMP '2020-01-01' + g * INTERVAL '1 second') AS ts,
       (TIMESTAMPTZ '2020-01-01 00:00:00+00' + g * INTERVAL '1 second') AS tstz,
       (DATE '2020-01-01' + (g % 3650)::INTEGER) AS d,
       (g % 2 = 0) AS b,
       ('name-' || g)::TEXT AS s,
       md5(g::TEXT) AS s2
FROM generate_series(1, {n}) AS t(g)
"""
DATASETS = ["narrow_100k", "narrow_1m", "wide_100k", "wide_1m"]
SIZES = {
    "narrow_100k": 100_000,
    "narrow_1m": 1_000_000,
    "wide_100k": 100_000,
    "wide_1m": 1_000_000,
}

DUCK_PATH = "bench.duckdb"
con = duckdb.connect(DUCK_PATH)
for name, n in SIZES.items():
    sql = NARROW if name.startswith("narrow") else WIDE
    con.execute(f"CREATE OR REPLACE TABLE {name} AS {sql.format(n=n)}")
    print("duckdb", name, con.execute(f"SELECT count(*) FROM {name}").fetchone()[0])
con.close()


## `%sql` over SQLAlchemy and ADBC

A short preview of the SQLAlchemy path, then the same query over ADBC with `.ArrowTable()` and `.PolarsDataFrame()`. The timed comparison comes next.


In [ ]:
%load_ext sql
%config SqlMagic.feedback = 0
%config SqlMagic.displaycon = False
%config SqlMagic.displaylimit = 5

%sql duckdb:///bench.duckdb --alias duck_sa
%sql duck_sa
%sql SELECT * FROM narrow_100k ORDER BY id


In [ ]:
from sql.connection import ConnectionManager
import adbc_driver_duckdb.dbapi

ConnectionManager.close_all()
duck_adbc = adbc_driver_duckdb.dbapi.connect(path="bench.duckdb")
%sql duck_adbc --alias duck_adbc
%sql duck_adbc
rs = %sql SELECT * FROM narrow_100k ORDER BY id
print(type(rs.sqlaproxy).__name__, rs.ArrowTable().schema)
rs.ArrowTable().slice(0, 3)


In [ ]:
df = rs.PolarsDataFrame()
df.head(3)


## Benchmark: `.PolarsDataFrame()` on both paths

Each case does 1 warmup round and 5 timed rounds; the table reports the median. The comparison is the current SQLAlchemy path versus the new ADBC Arrow path, both on DuckDB.


In [ ]:
import statistics
import time

import polars as pl
from sql.connection import ConnectionManager

ROUNDS = 5
WARMUP = 1


def cleanup():
    try:
        ConnectionManager.close_all()
    except Exception:
        pass


def time_polars(query):
    ip = get_ipython()

    def once():
        result = ip.run_line_magic("sql", query)
        height = result.PolarsDataFrame().height
        ConnectionManager.current._result_sets.close_all()
        return height

    rows = once()
    for _ in range(max(0, WARMUP - 1)):
        once()
    times = []
    for _ in range(ROUNDS):
        start = time.perf_counter()
        once()
        times.append(time.perf_counter() - start)
    return rows, statistics.median(times)


def connect(path):
    ip = get_ipython()
    cleanup()
    if path == "sqlalchemy":
        ip.run_line_magic("sql", f"duckdb:///{DUCK_PATH} --alias duck_sa")
        return None
    import adbc_driver_duckdb.dbapi as drv

    raw = drv.connect(path=DUCK_PATH)
    ip.user_ns["adbcconn"] = raw
    ip.run_line_magic("sql", "adbcconn --alias duck_adbc")
    return raw


records = []
for dataset in DATASETS:
    query = f"SELECT * FROM {dataset}"
    row = {"dataset": dataset}
    raw = None
    try:
        for path, key in (("sqlalchemy", "sqlalchemy_s"), ("adbc", "adbc_s")):
            raw = connect(path)
            rows, median_s = time_polars(query)
            row["rows"] = rows
            row[key] = median_s
            if raw is not None:
                raw.close()
                raw = None
            cleanup()
        row["speedup"] = row["sqlalchemy_s"] / row["adbc_s"]
        records.append(row)
        print(
            f"{dataset:12} rows={row['rows']:<8} "
            f"sqlalchemy={row['sqlalchemy_s']:.3f}s  "
            f"adbc={row['adbc_s']:.3f}s  "
            f"speedup={row['speedup']:.1f}x",
            flush=True,
        )
    finally:
        if raw is not None:
            try:
                raw.close()
            except Exception:
                pass
        cleanup()

bench = pl.DataFrame(records).select(
    "dataset",
    "rows",
    pl.col("sqlalchemy_s").round(3),
    pl.col("adbc_s").round(3),
    pl.col("speedup").round(1),
)
bench


In [ ]:
import matplotlib.pyplot as plt

labels = [r["dataset"] for r in records]
speedups = [r["speedup"] for r in records]

fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.bar(labels, speedups, color="#2563eb")
ax.set_ylabel("Speedup (×)")
ax.set_title("DuckDB: ADBC .PolarsDataFrame() vs SQLAlchemy .PolarsDataFrame()")
ax.axhline(1, color="0.5", linewidth=0.8)
for bar, value in zip(bars, speedups):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{value:.1f}x",
        ha="center",
        va="bottom",
        fontsize=9,
    )
fig.tight_layout()
fig.savefig("jupysql_adbc_speedup.png", dpi=120, bbox_inches="tight")
print("Wrote jupysql_adbc_speedup.png")
fig


## Parity and intervals

The same query is run on both paths. Interval columns should come back as `datetime.timedelta`, matching SQLAlchemy.


In [ ]:
from datetime import timedelta

PARITY_SQL = """
SELECT 42::INTEGER AS i,
       0.1::DOUBLE AS f,
       INTERVAL '1 day 2 hours 3 seconds' AS iv,
       DATE '2024-02-29' AS d,
       TRUE AS b,
       'héllo' AS s
"""


def collect(path, query):
    raw = connect(path)
    result = get_ipython().run_line_magic("sql", query)
    row = list(result)[0]
    keys = list(result.keys)
    types = {k: type(v).__name__ for k, v in zip(keys, row)}
    values = dict(zip(keys, row))
    if raw is not None:
        raw.close()
    cleanup()
    return values, types


sa_vals, sa_types = collect("sqlalchemy", PARITY_SQL)
adbc_vals, adbc_types = collect("adbc", PARITY_SQL)
parity_rows = []
for key in sa_vals:
    match = sa_vals[key] == adbc_vals[key] and sa_types[key] == adbc_types[key]
    flag = "OK" if match else "DIFF"
    print(
        f"{flag} {key:3}  sqlalchemy={sa_types[key]} {sa_vals[key]!r}  "
        f"adbc={adbc_types[key]} {adbc_vals[key]!r}"
    )
    parity_rows.append(
        {
            "column": key,
            "match": match,
            "sqlalchemy_type": sa_types[key],
            "adbc_type": adbc_types[key],
            "sqlalchemy": repr(sa_vals[key]),
            "adbc": repr(adbc_vals[key]),
        }
    )

assert isinstance(sa_vals["iv"], timedelta), sa_types["iv"]
assert isinstance(adbc_vals["iv"], timedelta), adbc_types["iv"]
assert sa_vals["iv"] == adbc_vals["iv"] == timedelta(days=1, hours=2, seconds=3)

pl.DataFrame(parity_rows)


## `.ArrowTable()` on a wider result


In [ ]:
raw = connect("adbc")
rs = %sql SELECT * FROM wide_100k
table = rs.ArrowTable()
print(type(table).__name__, table.num_rows, "rows")
print(table.schema)
if raw is not None:
    raw.close()
cleanup()
table.slice(0, 3)
